<a href="https://colab.research.google.com/github/JeissonCu/TareasPucp/blob/main/Tarea_4_Grupo_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

GRUPO 1 - INTEGRANTES:

* Estefania Chauca Alarcón
* Jose Luis Jr Vargas Rodas
* Valeria Arias Mosquito
* Isabel Salazar Mendoza
* Jarinson Castro Vargas
* Jeisson Enrique Cueva Caro

# SMOTE and ADASYN: Synthetic Oversampling Techniques for Imbalanced Data

This notebook provides a detailed explanation of **SMOTE** (Synthetic Minority Over-sampling Technique) and **ADASYN** (Adaptive Synthetic Sampling)—two powerful techniques for handling class imbalance in machine learning.

## 1. The Problem: Class Imbalance

In many real-world classification problems, we encounter **imbalanced datasets** where one class (the *minority* class) has significantly fewer samples than another (the *majority* class).

**Examples:**
- Fraud detection: 99.9% legitimate transactions, 0.1% fraud
- Medical diagnosis: rare diseases vs. healthy patients
- Spam detection: few spam emails among many legitimate ones

**Why is this a problem?**
- Standard classifiers optimize for overall accuracy, which can be achieved by predicting the majority class for everything
- The model may never learn the minority class patterns
- Metrics like accuracy become misleading (e.g., 99.9% accuracy by predicting "not fraud" always)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')
sns.set_palette("husl")

### Creating an Imbalanced Dataset for Demonstration

In [ ]:
# Create imbalanced dataset: 950 majority, 50 minority (95:5 ratio)
X, y = make_classification(n_samples=1000, n_features=2, n_redundant=0, n_informative=2,
                            n_clusters_per_class=1, weights=[0.95, 0.05], random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

print("Training set class distribution:")
print(pd.Series(y_train).value_counts().sort_index())
print(f"\nMinority class: {np.sum(y_train == 1)} samples ({100*np.mean(y_train):.1f}%)")
print(f"Majority class: {np.sum(y_train == 0)} samples ({100*(1-np.mean(y_train)):.1f}%)")

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], c='steelblue', alpha=0.6, label='Majority (0)', s=50)
ax.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], c='coral', alpha=0.8, label='Minority (1)', s=80)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_title('Imbalanced Training Data')
ax.legend()
plt.tight_layout()
plt.show()

---

## 2. SMOTE: Synthetic Minority Over-sampling Technique

**SMOTE** was introduced by Chawla et al. (2002) and is one of the most widely used oversampling methods.

### Key Idea
Instead of simply **duplicating** minority samples (which leads to overfitting), SMOTE creates **synthetic** samples by interpolating between existing minority instances. This expands the decision region for the minority class in a more meaningful way.

### The SMOTE Algorithm (Step by Step)

For each minority sample $x_i$:

1. **Find k nearest neighbors** of $x_i$ among other minority samples (typically k=5)

2. **Randomly select one neighbor** $x_{nn}$ from the k neighbors

3. **Create a synthetic sample** along the line segment between $x_i$ and $x_{nn}$:

   $$x_{new} = x_i + \lambda \cdot (x_{nn} - x_i)$$

   where $\lambda$ is a random number in $(0, 1)$. This places the new point somewhere between the two original points.

4. **Repeat** until the desired minority:majority ratio is achieved

### Visual Intuition

SMOTE places new synthetic points *between* existing minority points. In the plots below, synthetic samples are shown as **green "+" markers**, so you can see exactly where they were generated:

- **Original:** Few minority samples surrounded by many majority samples
- **After SMOTE:** Synthetic samples (+) fill the gaps between minority samples
- Result: A denser, more robust decision boundary for the minority class

### SMOTE Parameters

| Parameter | Description | Typical Value |
|-----------|-------------|---------------|
| `k_neighbors` | Number of nearest neighbors to consider | 5 |
| `sampling_strategy` | Target ratio (e.g., 'auto' for balance, or float) | 'auto' or 0.5 |
| `random_state` | Reproducibility | 42 |

### Advantages
- Reduces overfitting compared to random oversampling (no exact duplicates)
- Creates diverse synthetic samples in the feature space
- Simple and effective for many problems

### Limitations
- Treats all minority samples equally (may over-sample in already dense regions)
- Can create synthetic samples in majority-class regions (noise)
- Assumes the feature space is continuous (needs adaptation for categorical data: SMOTE-NC)

In [ ]:
# Install imbalanced-learn if needed: pip install imbalanced-learn
try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    !pip install imbalanced-learn
    from imblearn.over_sampling import SMOTE

# Apply SMOTE
smote = SMOTE(k_neighbors=5, random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("After SMOTE - Training set class distribution:")
print(pd.Series(y_train_smote).value_counts().sort_index())
print(f"\nTotal samples: {len(y_train_smote)} (was {len(y_train)})")

# In imblearn, fit_resample returns the ORIGINAL samples first, followed by
# the SYNTHETIC samples appended at the end. We use this to identify them:
X_synth_smote = X_train_smote[len(y_train):]
print(f"Synthetic samples generated by SMOTE: {len(X_synth_smote)}")

# Visualize SMOTE result: synthetic points shown with a distinct marker (+)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], c='steelblue', alpha=0.6, label='Majority', s=50)
axes[0].scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], c='coral', alpha=0.8, label='Minority (original)', s=80)
axes[0].set_title('Before SMOTE')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].legend()

axes[1].scatter(X_train_smote[y_train_smote==0, 0], X_train_smote[y_train_smote==0, 1], c='steelblue', alpha=0.6, label='Majority', s=50)
axes[1].scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], c='coral', alpha=0.8, label='Minority (original)', s=80)
axes[1].scatter(X_synth_smote[:, 0], X_synth_smote[:, 1], c='green', marker='+', s=70, label='Minority (synthetic)')
axes[1].set_title('After SMOTE')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].legend()

plt.tight_layout()
plt.show()

---

## 3. ADASYN: Adaptive Synthetic Sampling

**ADASYN** (He et al., 2008) is an *adaptive* extension of SMOTE. Instead of treating all minority samples equally, it focuses synthetic sample generation on **hard-to-learn** minority instances—those that are surrounded by majority samples or lie near the decision boundary.

### How ADASYN Differs from SMOTE

| Aspect | SMOTE | ADASYN |
|--------|-------|--------|
| **Strategy** | Uniform: same # of synthetics per minority sample | Adaptive: more synthetics for "difficult" samples |
| **Focus** | All minority samples equally | Borderline/hard-to-classify minority samples |
| **Weight** | No weighting | Each minority sample gets a weight ∝ local imbalance |
| **Result** | Even distribution of synthetic samples | Denser synthetic samples near decision boundary |

### The ADASYN Algorithm (He et al., 2008)

**Inputs / parameters defined in the paper:** $d_{th}$ = maximum tolerated degree of class imbalance (the paper uses $d_{th} = 0.75$), and $\beta \in [0, 1]$ = desired balance level after generation ($\beta = 1$ means a fully balanced dataset).

1. **Compute the degree of class imbalance** $d = m_s / m_l$ (minority over majority count). ADASYN only runs if $d < d_{th}$.

2. **Calculate the total number of synthetic samples** to generate: $G = (m_l - m_s) \times \beta$

3. **For each minority sample** $x_i$, find its K nearest neighbors (in the whole dataset) and compute $r_i = \Delta_i / K$, where $\Delta_i$ is the number of majority samples among those neighbors (higher $r_i$ = harder to classify)

4. **Normalize** $r_i$ to get a density distribution $\hat{r}_i$ that sums to 1

5. **For each minority sample**, generate $g_i = \hat{r}_i \times G$ synthetic samples

6. **Generate each synthetic sample** using the same interpolation as SMOTE:

   $$s_i = x_i + \lambda \cdot (x_{zi} - x_i)$$

   where $\lambda$ is a random number in $[0, 1]$ (note: the ADASYN paper defines the *closed* interval $[0,1]$, unlike SMOTE's $(0,1)$).

> **Note:** because the per-sample counts $g_i$ are rounded and depend on the adaptive weights, ADASYN does **not** guarantee an exact 50/50 balance — the final minority count is approximately, not exactly, equal to the majority count. Don't be surprised if `value_counts()` shows slightly different numbers. In `imblearn`, the balance target is controlled via `sampling_strategy` (the equivalent of $\beta$).

### When to Use ADASYN vs SMOTE

- **Use SMOTE** when minority samples are relatively well-distributed; you want a simple, uniform boost
- **Use ADASYN** when the minority class has complex structure, with some samples in "danger zones" (surrounded by majority); ADASYN will concentrate synthetic samples there

In [ ]:
from imblearn.over_sampling import ADASYN

# Apply ADASYN
adasyn = ADASYN(n_neighbors=5, random_state=42)
X_train_adasyn, y_train_adasyn = adasyn.fit_resample(X_train, y_train)

print("After ADASYN - Training set class distribution:")
print(pd.Series(y_train_adasyn).value_counts().sort_index())
print(f"\nTotal samples: {len(y_train_adasyn)}")
print("(Note: ADASYN's final balance is approximate, not an exact 50/50)")

# Synthetic samples are appended after the original ones
X_synth_adasyn = X_train_adasyn[len(y_train):]

# Compare: Original vs SMOTE vs ADASYN (synthetic points as green +)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], c='steelblue', alpha=0.6, s=50)
axes[0].scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], c='coral', alpha=0.8, s=80)
axes[0].set_title('Original (Imbalanced)')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

axes[1].scatter(X_train_smote[y_train_smote==0, 0], X_train_smote[y_train_smote==0, 1], c='steelblue', alpha=0.6, s=50)
axes[1].scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], c='coral', alpha=0.8, s=60)
axes[1].scatter(X_synth_smote[:, 0], X_synth_smote[:, 1], c='green', marker='+', s=60)
axes[1].set_title('After SMOTE (Uniform)')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

axes[2].scatter(X_train_adasyn[y_train_adasyn==0, 0], X_train_adasyn[y_train_adasyn==0, 1], c='steelblue', alpha=0.6, s=50)
axes[2].scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], c='coral', alpha=0.8, s=60)
axes[2].scatter(X_synth_adasyn[:, 0], X_synth_adasyn[:, 1], c='green', marker='+', s=60)
axes[2].set_title('After ADASYN (Adaptive)')
axes[2].set_xlabel('Feature 1')
axes[2].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

---

## 4. Practical Comparison: Model Performance

Following He et al. (2008), we evaluate not only the F1-score but also the **G-mean** (geometric mean of the per-class accuracies). G-mean is the metric on which ADASYN won across all five datasets in the original paper: a high G-mean means the model improves minority-class accuracy **without sacrificing** the majority class.

In [ ]:
from imblearn.metrics import geometric_mean_score

def train_and_evaluate(X_tr, y_tr, name):
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_test)
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=['Majority', 'Minority']))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    f1 = f1_score(y_test, y_pred, pos_label=1)
    gmean = geometric_mean_score(y_test, y_pred)
    print(f"G-mean: {gmean:.3f}")
    return f1, gmean

f1_original, gm_original = train_and_evaluate(X_train, y_train, 'Original (No Resampling)')
f1_smote, gm_smote = train_and_evaluate(X_train_smote, y_train_smote, 'SMOTE')
f1_adasyn, gm_adasyn = train_and_evaluate(X_train_adasyn, y_train_adasyn, 'ADASYN')

# Comparison: F1 (minority) and G-mean
methods = ['Original', 'SMOTE', 'ADASYN']
f1_scores = [f1_original, f1_smote, f1_adasyn]
gm_scores = [gm_original, gm_smote, gm_adasyn]
colors = ['coral', 'steelblue', 'seagreen']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bars0 = axes[0].bar(methods, f1_scores, color=colors)
axes[0].set_ylabel('F1-Score (Minority Class)')
axes[0].set_title('Minority-Class F1 by Method')
axes[0].set_ylim(0, 1)
for bar, score in zip(bars0, f1_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{score:.3f}', ha='center', fontsize=11)

bars1 = axes[1].bar(methods, gm_scores, color=colors)
axes[1].set_ylabel('G-mean')
axes[1].set_title('G-mean by Method (metric emphasized in He et al., 2008)')
axes[1].set_ylim(0, 1)
for bar, score in zip(bars1, gm_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{score:.3f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

---

## 5. Summary and Best Practices

### Key Takeaways

1. **SMOTE** creates synthetic minority samples via linear interpolation between neighbors. It is uniform and simple.

2. **ADASYN** adapts by generating more synthetic samples for minority instances that are harder to classify (near the boundary). Its final balance is approximate, controlled by $\beta$ (`sampling_strategy` in imblearn).

3. **There is a recall/precision trade-off.** In the original ADASYN paper (Table 2), the plain decision tree without resampling achieved the best *precision* on all five datasets, while the oversampling methods won on *recall*, *F-measure*, and *G-mean*. Oversampling shifts the decision boundary toward the minority class: you catch more true minority cases (higher recall) at the cost of more false positives (lower precision). Choose based on the cost of each error type in your application.

4. **Always apply resampling only to the training set**, never to the test set, to avoid data leakage.

5. **Use pipelines** (e.g., `imblearn.pipeline.Pipeline`) so that resampling happens within cross-validation folds correctly.

In [ ]:
# Example: Using a pipeline (recommended for proper CV)
from imblearn.pipeline import Pipeline as ImbPipeline

pipeline = ImbPipeline([
    ('smote', SMOTE(k_neighbors=5, random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred_pipe = pipeline.predict(X_test)
print("Pipeline (SMOTE + LogisticRegression) - Classification Report:")
print(classification_report(y_test, y_pred_pipe, target_names=['Majority', 'Minority']))

---

## 6. Exporting the Notebook (Colab)

Google Drive must be mounted **before** running `nbconvert`, since the notebook path points to Drive. (`nbconvert` comes pre-installed in Colab, so no `pip install` is needed.)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Replace with your actual notebook path in Drive
notebook_name = "/content/drive/MyDrive/Colab Notebooks/SMOTE_and_ADASYN_Explained.ipynb"

# Convert the notebook to HTML
!jupyter nbconvert --to html "{notebook_name}"

### Referencias

- Chawla, N. V., Hall, L. O., Bowyer, K. W., & Kegelmeyer, W. P. (2002). "SMOTE: Synthetic Minority Over-sampling Technique." *Journal of Artificial Intelligence Research*, 16, 321–357.
- He, H., Bai, Y., Garcia, E. A., & Li, S. (2008). "ADASYN: Adaptive synthetic sampling approach for imbalanced learning." *IEEE International Joint Conference on Neural Networks (IJCNN 2008)*, pp. 1322–1328.

---

## Conclusiones: correcciones realizadas en esta versión

Esta versión del notebook incorpora las siguientes correcciones y mejoras respecto a la versión original, contrastadas con el paper original de ADASYN (He et al., 2008):


1. **Visualización de los puntos sintéticos.** El texto prometía distinguir los puntos sintéticos con un marcador especial, pero los gráficos los mostraban del mismo color que los originales. Ahora los sintéticos se grafican como cruces verdes (+), aprovechando que en `imblearn` el resultado de `fit_resample` devuelve primero las muestras originales y luego las sintéticas. Se eliminó también el código muerto (`n_original_minority`, `minority_indices`) que se calculaba sin usarse.

2. **Notación de λ ajustada al paper.** Para ADASYN, el paper define λ ∈ [0, 1] (intervalo cerrado, Ec. 5), no (0, 1). Se mantuvo (0, 1) para SMOTE, que es la convención de Chawla et al. (2002).

3. **Algoritmo de ADASYN completado.** Añadimos dos parámetros que el paper define explícitamente y que faltaban: el umbral de desbalance d_th (el paper usa 0.75; ADASYN solo se ejecuta si d = ms/ml < d_th) y el coeficiente β ∈ [0, 1] que fija el nivel de balance deseado mediante G = (ml − ms) × β. Se agregó además la aclaración de que ADASYN no garantiza un balance exacto 50/50 (los gᵢ se redondean), y que en `imblearn` el equivalente de β es `sampling_strategy`.

4. **Métrica G-mean incorporada.** El paper evalúa con Precision, Recall, F-measure y G-mean, y destaca que ADASYN ganó en G-mean en los cinco datasets (mejora la clase minoritaria sin sacrificar la mayoritaria). La sección 4 ahora calcula y grafica el G-mean junto al F1.

5. **Trade-off recall/precision documentado.** El modelo sin resampling ganó en *precision* en los cinco datasets, mientras que el oversampling mejora *recall*, *F-measure* y *G-mean*. Es el mismo patrón que se observa en la comparación práctica de este notebook.

6. **Incluimos la referencia.** Se incluyó la cita completa del paper de ADASYN con autores, conferencia y páginas (pp. 1322–1328, IJCNN 2008).